# ACT Regression (From-Scratch NN)

## Step 1: Load Data
This notebook loads the EdGap dataset from the original URL and reproduces the preprocessing logic before we build a NumPy-based neural network.


In [ ]:
import pandas as pd
import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
import time

import warnings
warnings.filterwarnings("ignore", category=UserWarning, message="Unknown extension is not supported and will be removed")

sns.set_style("whitegrid")

df = pd.read_excel(
    "https://raw.githubusercontent.com/brian-fischer/DATA-5100/main/EdGap_data.xlsx",
    dtype={"NCESSCH School ID": object},
)
df.head()


## Step 2: Rename, Clean, and Impute
Rename columns to match the notebook, mark invalid values as missing, and fill missing predictors with their mode. Outputs number of missing values in each column as a check.


In [ ]:
df = df.rename(columns={
    "NCESSCH School ID": "id",
    "CT Pct Adults with College Degree": "percent_college",
    "CT Unemployment Rate": "rate_unemployment",
    "CT Pct Childre In Married Couple Family": "percent_married",
    "CT Median Household Income": "median_income",
    "School ACT average (or equivalent if SAT score)": "average_act",
    "School Pct Free and Reduced Lunch": "percent_lunch",
})

df.loc[df["average_act"] < 1, "average_act"] = np.nan
df.loc[df["percent_lunch"] < 0, "percent_lunch"] = np.nan

input_modes = df[df.columns.difference(["id", "average_act"])].mode().iloc[0]
df = df.fillna(input_modes)
df = df.dropna()
df.isna().sum()


## Step 3: Split and Standardize
Split into train/test and use `StandardScaler` to put features on the same scale (mean 0, std 1). This helps neural nets train more reliably.


In [ ]:
X = df[df.columns.difference(["id", "average_act"]) ]
y = df["average_act"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=4242)

scaler = StandardScaler().fit(X_train)
X_train = scaler.transform(X_train)
X_test = scaler.transform(X_test)

## Step 4: From-Scratch NN (JAX) — Hyperparameters
Define the model size and training settings before building the network.


In [ ]:
# Hyperparameters
input_dim = X_train.shape[1]
hidden_dim = 100
output_dim = 1

# Shared training settings
epochs = 1000
seed = 0
shuffle = True

# Optimizer mode and batch style controls
# full_gd: full-batch GD
# mini_gd: mini-batch GD
# momentum: momentum path with explicit batch mode
optimizer_mode = "momentum"  # options: "full_gd", "mini_gd", "momentum"
momentum_batch_style = "mini"  # options (only for momentum): "mini", "full"

# Batching and optimizer hyperparameters
mini_batch_size = 64
momentum_batch_size = 128

full_gd_lr = 5e-4
mini_gd_lr = 5e-4
momentum_lr = 2e-4
momentum_beta = 0.9

if optimizer_mode == "full_gd":
    lr = full_gd_lr
    batch_size = X_train.shape[0]
    shuffle = False
    use_momentum = False
    active_mode = "full_gd (Full Batch)"
elif optimizer_mode == "mini_gd":
    lr = mini_gd_lr
    batch_size = mini_batch_size
    use_momentum = False
    active_mode = "Mini-Batch GD"
elif optimizer_mode == "momentum":
    lr = momentum_lr
    if momentum_batch_style not in {"mini", "full"}:
        raise ValueError(f"momentum_batch_style must be 'mini' or 'full', got {momentum_batch_style!r}")
    if momentum_batch_style == "full":
        batch_size = X_train.shape[0]
        shuffle = False
        active_mode = "Momentum (Full Batch)"
    else:
        batch_size = momentum_batch_size
        active_mode = "Momentum (Mini-Batch)"
    use_momentum = True
else:
    raise ValueError(f"Unknown optimizer_mode {optimizer_mode!r}; expected 'full_gd', 'mini_gd', or 'momentum'")


### Step 4.1: Initialize Weights and Biases
We start with small random weights and zero biases for the neurons to break symmetry and keep early activations stable. Each neuron produces learned feature, and all of those will feed into the final output to compute score.


In [ ]:
# Initialize flexible network parameters (JAX)
# Example architecture: input -> 64 -> 32 -> 1
layer_sizes = [input_dim, 64, 32, 1]


def init_network_params(layer_sizes, seed=2999):
    """Initialize one weight matrix and one bias vector per layer."""
    key = jax.random.PRNGKey(seed)
    params = []

    for idx in range(len(layer_sizes) - 1):
        key, subkey = jax.random.split(key)
        W = jax.random.normal(subkey, (layer_sizes[idx], layer_sizes[idx + 1])) * 0.01
        b = jnp.zeros((1, layer_sizes[idx + 1]))
        params.append({"W": W, "b": b})

    return params


params = init_network_params(layer_sizes, seed=2999)

# Checkpoint: print initialized shapes
for i, layer in enumerate(params):
    print(f"Layer {i + 1}: W {layer['W'].shape}, b {layer['b'].shape}")


### Step 4.2: Activation Functions
Define ReLU and its gradient for the hidden layer.


In [ ]:
def relu(x):
    return jnp.maximum(0, x)


### Step 4.3: Prepare Data for JAX and Define Forward/Loss
Convert arrays to `jnp` and define the forward pass plus MSE loss. Forward takes inputs and produces predictions using current weights. MSE loss measures how wrong the predictions are compared to true values. Returns single number that we try to minimize during training.


In [ ]:
# Convert data to JAX arrays
X_train_j = jnp.array(X_train)
X_test_j = jnp.array(X_test)
y_train_j = jnp.array(y_train.to_numpy()).reshape(-1, 1) # Reshape to (num_samples, 1) for regression targets
y_test_j = jnp.array(y_test.to_numpy()).reshape(-1, 1) # Reshape to (num_samples, 1) for regression targets

def forward(params, X):
    """Flexible forward pass for an arbitrary depth MLP.

    Hidden layers use ReLU and the final layer is linear (regression).
    """
    activations = X
    num_layers = len(params)

    for layer_idx, layer in enumerate(params):
        W = layer["W"]
        b = layer["b"]
        z = activations @ W + b

        if layer_idx < num_layers - 1:
            activations = relu(z)
        else:
            activations = z

    return activations

def mse_loss(params, X, y):
    y_pred = forward(params, X)
    return jnp.mean((y_pred - y) ** 2)

# Set up parameters and gradient function using loss
grad_fn = jax.jit(jax.grad(mse_loss))


### Step 4.4: Training Loop (JAX)
Train the network with mini-batch gradient descent over epochs. At each epoch, optionally shuffle training data, split into mini-batches, compute gradients on each batch using jax.grad, and update parameters with full_gdl_gd. After processing all batches in an epoch, compute and store epoch-level training loss on the full training set. After training, evaluate on the test set and compare predicted vs actual values using test MSE.


In [ ]:
# Reusable full_gd update helper

def full_gd_update(params, grads, lr):
    """Apply vanilla full_gd to the list-of-dict parameter structure."""
    return [
        {
            "W": layer["W"] - lr * grad["W"],
            "b": layer["b"] - lr * grad["b"],
        }
        for layer, grad in zip(params, grads)
    ]

def init_velocities(params):
    # Create zero-initialized velocity tensors matching each layer's W and b shapes.
    # These store the running update direction for momentum.
    return [
        {
            "W": jnp.zeros_like(layer["W"]),
            "b": jnp.zeros_like(layer["b"]),
        }
        for layer in params
    ]

def momentum_update(params, grads, velocities, lr, momentum_beta):
    # One momentum step:
    # 1) update velocity from past velocity + current gradient
    # 2) update parameters using velocity
    new_params = []
    new_velocities = []
    for layer, grad, vel in zip(params, grads, velocities):
        vW = momentum_beta * vel["W"] + grad["W"]
        vb = momentum_beta * vel["b"] + grad["b"]


        W = layer["W"] - lr * vW
        b = layer["b"] - lr * vb

        new_params.append({"W": W, "b": b})
        new_velocities.append({"W": vW, "b": vb})
    return new_params, new_velocities

# Training loop with explicit full-batch and mini-batch paths
loss_history = np.zeros(epochs, dtype=float)
test_loss_history = np.zeros(epochs, dtype=float)
step_losses = []
rng = jax.random.PRNGKey(seed)
n = X_train_j.shape[0]
is_full_batch_mode = (batch_size == X_train_j.shape[0])

train_start = time.perf_counter()

velocities = init_velocities(params) if use_momentum else None
for epoch in range(epochs):
    if is_full_batch_mode:
        grads = grad_fn(params, X_train_j, y_train_j)
        if use_momentum:
            params, velocities = momentum_update(params, grads, velocities, lr, momentum_beta)
        else:
            params = full_gd_update(params, grads, lr)
        step_losses.append(float(mse_loss(params, X_train_j, y_train_j)))
    else:
        if shuffle:
            rng, perm_key = jax.random.split(rng)
            perm = jax.random.permutation(perm_key, n)
            X_epoch = X_train_j[perm]
            y_epoch = y_train_j[perm]
        else:
            X_epoch = X_train_j
            y_epoch = y_train_j

        for start in range(0, n, batch_size):
            X_batch = X_epoch[start:start + batch_size]
            y_batch = y_epoch[start:start + batch_size]

            grads = grad_fn(params, X_batch, y_batch)
            if use_momentum:
                params, velocities = momentum_update(params, grads, velocities, lr, momentum_beta)
            else:


                
                params = full_gd_update(params, grads, lr)
            step_losses.append(float(mse_loss(params, X_batch, y_batch)))

    loss_history[epoch] = float(mse_loss(params, X_train_j, y_train_j))
    test_loss_history[epoch] = float(mse_loss(params, X_test_j, y_test_j))

training_runtime_seconds = time.perf_counter() - train_start

# Backwards compatibility with older references in notebook
losses = loss_history

final_loss = loss_history[-1]
print(f"Training complete after {epochs} epochs. Final train loss: {final_loss:.4f}")


### Step 4.5: Evaluate and Plot
After model is trained and parameters are adjusted, use the remaining 20% test data to test the network on the test set, compute test MSE, and plot predicted vs actual. 


In [ ]:
y_test_pred = forward(params, X_test_j)

y_true = np.array(y_test_j).flatten()
y_pred = np.array(y_test_pred).flatten()
mse = mean_squared_error(np.array(y_test_j), np.array(y_test_pred))

later_med = np.median(losses[10:])          # stable reference
threshold = 3.0 * later_med                 # "too large" cutoff

# first epoch index that is not an extreme spike
valid_idx = np.where(losses <= threshold)[0]
start_epoch = int(valid_idx[0]) if len(valid_idx) else 0

fig = plt.figure(figsize=(8, 5))
ax = fig.add_subplot()
y_plot = losses[start_epoch:]
ax.plot(np.arange(start_epoch, epochs), y_plot, label="Train MSE")
ax.set_ylim(y_plot.min(), y_plot.max())
ax.set_xlabel("Epoch")
ax.set_ylabel("MSE")
opt_name = active_mode
ax.set_title(f"Training Loss vs Epoch ({opt_name})\nFinal Test MSE: {mse:.4f}")
ax.legend()
ax.grid(True)

plt.tight_layout()
print(f"Training runtime ({opt_name}): {training_runtime_seconds:.2f} seconds")
